In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import glob
import zipfile
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 36
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## ODIN Pipeline

**Source:** Open Data Inventory (Open Data Watch)
**Access:** Manual ZIP download (per-year Excel files); auto-detects ZIP in Downloads
**Download instructions:** See `docs/instructions_data_maintenance.md` — ODIN section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| ODIN Overall Open Data Index (coverage + openness) | Statistical capacity / govt transparency | Primary tier 1 |
| Coverage and openness subscores | Statistical capacity | Supporting detail |

### Note
ODIN scores national statistical systems on data coverage and openness, 0-100.
Biennial editions stacked into a country-year panel.

In [2]:
# Auto-detect the ODIN ZIP in Downloads. The filename ("2016-2024 data.zip") has a year range
# that changes per edition, so we glob broadly on "*data*.zip" then VALIDATE contents:
# a genuine ODIN ZIP contains per-year Excel files named like a 4-digit year (e.g. 2016.xlsx).
import re

odin_candidates = glob.glob(os.path.join(DOWNLOADS_DIR, "*data*.zip"))

def is_odin_zip(path):
    """A genuine ODIN ZIP contains Excel files named as 4-digit years (e.g. '2016.xlsx')."""
    try:
        with zipfile.ZipFile(path) as z:
            names = z.namelist()
        year_xlsx = [n for n in names if re.fullmatch(r'.*?(\d{4})\.xlsx', os.path.basename(n))]
        return len(year_xlsx) >= 2  # at least two yearly editions
    except zipfile.BadZipFile:
        return False

odin_zips = [p for p in odin_candidates if is_odin_zip(p)]

if not odin_zips:
    print(f"No valid ODIN ZIP found in {DOWNLOADS_DIR}")
    print("Download complete data from odin.opendatawatch.com/data")
else:
    odin_zip = max(odin_zips, key=os.path.getmtime)
    print(f"Found ODIN ZIP: {os.path.basename(odin_zip)}")
    with zipfile.ZipFile(odin_zip) as z:
        members = [n for n in z.namelist() if n.endswith('.xlsx')]
    print(f"Year files inside: {members}")

Found ODIN ZIP: 2016-2024 data.zip
Year files inside: ['2016.xlsx', '2024.xlsx', '2022.xlsx', '2020.xlsx', '2018.xlsx', '2017.xlsx']


In [6]:
# Build ODIN country-year panel by aggregating category-level scores in each edition.
# The workbook has NO official country-level 0-100 index — only per-category element scores (0-10).
# We aggregate by SIMPLE MEAN across the 22 data categories within each country-year.
# NOTE: transparent aggregation of ODIN's published category data — NOT ODIN's official national
# index (which uses ODIN's own category/element weighting and 0-100 scaling).

COVERAGE_ELEMENTS = [
    'Indicator coverage and disaggregation', 'Data available last 5 years',
    'Data available last 10 years', 'First administrative level', 'Second administrative level',
]
OPENNESS_ELEMENTS = [
    'Machine readability', 'Non-proprietary', 'Download options',
    'Metadata availability', 'Terms of use',
]

def load_odin_edition(zip_path, member_name):
    """Load one ODIN edition's scores sheet (sheet 0); normalize headers (strip whitespace/newlines)."""
    with zipfile.ZipFile(zip_path) as z:
        raw = pd.read_excel(io.BytesIO(z.read(member_name)), sheet_name=0, engine='openpyxl')
    raw.columns = [str(c).strip() for c in raw.columns]
    return raw

# Discover all year editions in the ZIP (year parsed from filename — no hardcoding)
with zipfile.ZipFile(odin_zip) as z:
    edition_files = {int(re.search(r'(\d{4})\.xlsx', os.path.basename(n)).group(1)): n
                     for n in z.namelist() if n.endswith('.xlsx')}

frames = []
for edition_year, member in sorted(edition_files.items()):
    df = load_odin_edition(odin_zip, member)
    present_cov = [c for c in COVERAGE_ELEMENTS if c in df.columns]
    present_opn = [c for c in OPENNESS_ELEMENTS if c in df.columns]
    for c in present_cov + present_opn:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Mean across categories per element, then mean across elements -> dimension subscore per country
    cov_mean = df.groupby('Country code')[present_cov].mean().mean(axis=1)
    opn_mean = df.groupby('Country code')[present_opn].mean().mean(axis=1)

    out = pd.DataFrame({
        'country_code': cov_mean.index,
        'odin_coverage': cov_mean.values,
        'odin_openness': opn_mean.reindex(cov_mean.index).values,
    })
    out['odin_overall'] = out[['odin_coverage', 'odin_openness']].mean(axis=1)
    out['year'] = edition_year
    frames.append(out)

odin = pd.concat(frames, ignore_index=True)
odin = odin[['country_code', 'year', 'odin_coverage', 'odin_openness', 'odin_overall']]
odin = odin.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"ODIN panel shape: {odin.shape}")
print(f"Editions: {sorted(edition_files)}")
print(f"Countries: {odin['country_code'].nunique()}")
print(f"Score ranges — coverage: {odin['odin_coverage'].min():.2f}-{odin['odin_coverage'].max():.2f}, "
      f"openness: {odin['odin_openness'].min():.2f}-{odin['odin_openness'].max():.2f}")
print("\nSample (latest edition, top overall):")
latest_ed = odin[odin['year'] == odin['year'].max()]
print(latest_ed.sort_values('odin_overall', ascending=False).head(8).to_string(index=False))

ODIN panel shape: (1110, 5)
Editions: [2016, 2017, 2018, 2020, 2022, 2024]
Countries: 200
Score ranges — coverage: 0.02-1.53, openness: 0.02-1.91

Sample (latest edition, top overall):
country_code  year  odin_coverage  odin_openness  odin_overall
         DNK  2024       0.786765       0.931818      0.859291
         MYS  2024       0.736052       0.981818      0.858935
         FIN  2024       0.779189       0.936364      0.857776
         SGP  2024       0.760417       0.945455      0.852936
         POL  2024       0.725446       0.968182      0.846814
         NOR  2024       0.715909       0.950000      0.832955
         HKG  2024       0.737689       0.913636      0.825663
         OMN  2024       0.646791       0.972727      0.809759


In [7]:
# Data currency: latest edition year — derived from the editions present, no hardcoding
data_as_of = str(int(odin['year'].max()))

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "odin_clean.csv")
odin.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {odin.shape}")

n_countries = odin['country_code'].nunique()

# Update download log
update_entry(
    "ODIN",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of,
    local_filename="odin_clean.csv",
    latest_available_version=data_as_of,
    notes=("Open Data Inventory (Open Data Watch). Statistical-system coverage & openness. "
           "Manual ZIP of per-edition Excels, auto-detected in Downloads (validated by contents). "
           "Sub-scores (odin_coverage, odin_openness, odin_overall) are a TRANSPARENT SIMPLE-MEAN "
           "aggregation of ODIN's per-category element scores — NOT ODIN's official 0-100 national "
           "index (which uses ODIN's own weighting/scaling). RAW relative scale (~0-2), not rescaled; "
           "ranking is valid, absolute values for downstream normalization only. "
           "Biennial editions stacked. Overlaps substantially with IMF SPI (statistical capacity). "
           f"Coverage: {n_countries} countries.")
)
print_entry("ODIN")

Written: C:\Users\mjbou\governance-framework\data\processed\odin_clean.csv
Shape: (1110, 5)
[download_log] Updated entry for ODIN
  source_id: ODIN
  last_attempted_date: 2026-06-17
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2024
  local_filename: odin_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: Open Data Inventory (Open Data Watch). Statistical-system coverage & openness. Manual ZIP of per-edition Excels, auto-detected in Downloads (validated by contents). Sub-scores (odin_coverage, odin_openness, odin_overall) are a TRANSPARENT SIMPLE-MEAN aggregation of ODIN's per-category element scores — NOT ODIN's official 0-100 national index (which uses ODIN's own weighting/scaling). RAW relative scale (~0-2), not rescaled; ranking is valid, absolute values for downstream normalization only. Biennial editions stacked. Overlaps substantially with IMF SPI (statistical capacity). Coverage: 200 countries.


Scores sheet shape: (4338, 17)
Columns: ['Year', 'Region', 'Region code', 'Country', 'Country code', 'Data category', 'Indicator coverage and disaggregation', 'Data available last 5 years', 'Data available last 10 years', 'First administrative level\n', 'Second administrative level\n', 'Machine readability', 'Non-proprietary', 'Download options\n', 'Metadata availability', 'Terms of use\n', 'Overall score']

Rows per country (sample):
Country
Afghanistan    22
Albania        22
Algeria        22
Andorra        22
Angola         22
dtype: int64

Distinct data categories: 22

Overall score range: 0.0 — 10.0

Sheets in this edition: ['Scores and subscores 2024', 'Datasets 2024', '2024 Metadata']
